# Chapter 1 — VLM Tokenizer Training

This notebook trains the **text tokenizer** for nanochat_vlm.

Key design choice:
- **Text tokens are learned via BPE**
- **Visual tokens are fixed, symbolic placeholders**
- **Images are NOT tokenized here**

This notebook only defines the *symbol space* shared by text and vision.

## LLM vs VLM Tokenizer

LLM:
    text → BPE → token IDs

VLM:
    [text tokens] + [visual placeholder tokens]

Important:
- Visual tokens do not come from data
- They do not correspond to UTF-8 bytes
- They are aligned later with vision encoder outputs


## Visual Special Tokens

These tokens reserve ID space for multimodal alignment.

They behave like atomic symbols:
- never split by BPE
- never counted toward bits-per-byte


In [ ]:
import os
import time
import torch
import argparse
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

from nanochat_vlm.tokenizer import RustBPETokenizer
from nanochat_vlm.common import get_base_dir
from nanochat_vlm.dataset import parquets_iter_batched

## Define Special Tokens

Text tokens are learned.
Visual tokens are fixed and injected before training.

In [ ]:
TEXT_SPECIAL_TOKENS = [
    "<pad>", "<bos>", "<eos>", "<unk>"
]

VISUAL_SPECIAL_TOKENS = [
    "<image>",
    "</image>",
    "<im_start>",
    "<im_end>",
    "<im_patch>"
]

SPECIAL_TOKENS = TEXT_SPECIAL_TOKENS + VISUAL_SPECIAL_TOKENS
NUM_VISUAL_TOKENS = len(VISUAL_SPECIAL_TOKENS)

## Tokenizer Hyperparameters

The total vocabulary includes:
- learned text tokens
- reserved visual tokens

We reduce the learned vocabulary accordingly.

In [ ]:
MAX_CHARS = 10_000_000_000
DOC_CAP = 10_000
VOCAB_SIZE_TOTAL = 65_536

TEXT_VOCAB_SIZE = VOCAB_SIZE_TOTAL - NUM_VISUAL_TOKENS

print("Total vocab size:", VOCAB_SIZE_TOTAL)
print("Text vocab size:", TEXT_VOCAB_SIZE)
print("Visual tokens:", NUM_VISUAL_TOKENS)

## Text Iterator

Images are not used here.
Only raw text is streamed into BPE training.

In [ ]:
def text_iterator():
    nchars = 0
    for batch in parquets_iter_batched(split="train"):
        for doc in batch:
            doc = doc[:DOC_CAP]
            nchars += len(doc)
            yield doc
            if nchars > MAX_CHARS:
                return

## Train Tokenizer

Visual tokens are injected as *special tokens*.
Only text statistics affect BPE merges.

In [ ]:
t0 = time.time()

tokenizer = RustBPETokenizer.train_from_iterator(
    iterator=text_iterator(),
    vocab_size=TEXT_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
)

train_time = time.time() - t0
print(f"Training time: {train_time:.2f}s")

## Save Tokenizer Artifacts

In [ ]:
base_dir = get_base_dir()
tokenizer_dir = os.path.join(base_dir, "tokenizer")
tokenizer.save(tokenizer_dir)

print("Saved tokenizer to:", tokenizer_dir)

## Sanity Checks

1. Text round-trip
2. Visual tokens are atomic

In [ ]:
text = "Hello world!"
assert tokenizer.decode(tokenizer.encode(text)) == text

vlm_text = "<im_start>Hello<im_end>"
assert tokenizer.decode(tokenizer.encode(vlm_text)) == vlm_text

print("Sanity checks passed.")

## Token Bytes Cache

Bits-per-byte is used instead of loss to compare models fairly.

Rules:
- UTF-8 text tokens → byte length
- Special tokens (including visual tokens) → 0 bytes

In [ ]:
vocab_size = tokenizer.get_vocab_size()
special_set = set(tokenizer.get_special_tokens())

token_bytes = []
for token_id in range(vocab_size):
    token_str = tokenizer.decode([token_id])
    if token_str in special_set:
        token_bytes.append(0)
    else:
        token_bytes.append(len(token_str.encode("utf-8")))

token_bytes = torch.tensor(token_bytes, dtype=torch.int32)
torch.save(token_bytes, os.path.join(tokenizer_dir, "token_bytes.pt"))

print("Saved token_bytes.pt")

## What This Enables

This tokenizer:
- defines a shared symbol space for text + vision
- guarantees stable token IDs
- cleanly separates language statistics from perception

Next chapters:
- vision encoder
- patch → <im_patch> alignment
- multimodal forward pass